In [1]:
import pandas as pd
import chromadb
import json
import time
import numpy as np

# 1. Load Ground Truth
gt_df = pd.read_excel(r'G:\Khoa_Luan\Source_code\backend\chunking\IUH_Retrival_Chatbot_GroundTruth.xlsx')
print(f"Đã tải {len(gt_df)} câu hỏi kiểm thử.")

# 2. Kết nối ChromaDB (Cho luồng Vector)
from chromadb.utils import embedding_functions

chroma_client = chromadb.PersistentClient(path=r"G:\Khoa_Luan\Source_code\data\chroma_db") 
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
) 

collection = chroma_client.get_collection(
    name="camnang_iuh_collection", 
    embedding_function=emb_fn
)
print("Khởi tạo ChromaDB thành công!")

Đã tải 28 câu hỏi kiểm thử.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Khởi tạo ChromaDB thành công!


Khởi tạo BM25 (Luồng Keyword) và BGE Re-ranker (Phễu lọc)

In [2]:
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
import os

# --- 1. KHỞI TẠO BM25 ---
# Để BM25 hoạt động, nó cần biết toàn bộ text trong database của bạn
print("Đang nạp toàn bộ dữ liệu từ ChromaDB để xây dựng index BM25...")
all_data = collection.get(include=['documents', 'metadatas'])
corpus_chunks = all_data['documents']
corpus_urls = [m.get('source', '') for m in all_data['metadatas']]

# Tokenize đơn giản bằng cách tách từ (có thể dùng thu viện xử lý tiếng việt tốt hơn nếu cần)
tokenized_corpus = [doc.lower().split(" ") for doc in corpus_chunks]
bm25 = BM25Okapi(tokenized_corpus)
print(f"Xây dựng xong BM25 Index với {len(corpus_chunks)} chunks.")

# --- 2. KHỞI TẠO LOCAL RE-RANKER ---
print("Đang tải/đọc mô hình BGE Reranker (Local)...")
local_reranker = CrossEncoder('BAAI/bge-reranker-v2-m3')
print("Khởi tạo BGE Reranker thành công!")

Đang nạp toàn bộ dữ liệu từ ChromaDB để xây dựng index BM25...
Xây dựng xong BM25 Index với 518 chunks.
Đang tải/đọc mô hình BGE Reranker (Local)...


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Khởi tạo BGE Reranker thành công!


Xây dựng các hàm Truy xuất và Xếp hạng chéo (RRF)

In [ ]:
def get_vector_results(query, k=10):
    """Lấy Top K chunk từ Vector Database."""
    results = collection.query(query_texts=[query], n_results=k)
    return results['documents'][0], [m.get('source', '') for m in results['metadatas'][0]]

def get_bm25_results(query, k=10):
    """Lấy Top K chunk bằng BM25 (Khớp từ khóa)."""
    tokenized_query = query.lower().split(" ")
    
    # Lấy index của Top K kết quả cao nhất
    top_n_indexes = np.argsort(bm25.get_scores(tokenized_query))[::-1][:k]
    
    retrieved_chunks = [corpus_chunks[i] for i in top_n_indexes]
    retrieved_urls = [corpus_urls[i] for i in top_n_indexes]
    return retrieved_chunks, retrieved_urls

def rrf_merge(list1_chunks, list1_urls, list2_chunks, list2_urls, k=60):
    """Gộp 2 danh sách kết quả bằng thuật toán Reciprocal Rank Fusion (RRF)."""
    rrf_scores = {}
    chunk_to_url = {}
    
    # Hàm tính điểm cho một danh sách
    def compute_rrf(chunks, urls):
        for rank, (chunk, url) in enumerate(zip(chunks, urls)):
            if chunk not in rrf_scores:
                rrf_scores[chunk] = 0
                chunk_to_url[chunk] = url
            # Công thức RRF tiêu chuẩn
            rrf_scores[chunk] += 1 / (k + rank + 1)
            
    compute_rrf(list1_chunks, list1_urls)
    compute_rrf(list2_chunks, list2_urls)
    
    # Sắp xếp lại theo tổng điểm RRF giảm dần
    sorted_items = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    
    merged_chunks = [item[0] for item in sorted_items]
    merged_urls = [chunk_to_url[item[0]] for item in sorted_items]
    
    return merged_chunks, merged_urls

def local_rerank(query, documents, urls, top_n=3):
    """Lọc lần cuối bằng BGE Reranker."""
    if not documents:
        return [], []
        
    pairs = [[query, doc] for doc in documents]
    scores = local_reranker.predict(pairs)
    
    scored_docs = list(zip(scores, documents, urls))
    scored_docs.sort(key=lambda x: x[0], reverse=True)
    
    top_chunks = [doc for score, doc, url in scored_docs[:top_n]]
    top_urls = [url for score, doc, url in scored_docs[:top_n]]
    
    return top_chunks, top_urls

In [4]:
def advanced_hybrid_rag(query, top_k_initial=10, top_k_final=3):
    """Quy trình chuẩn: Vector (Top 10) + BM25 (Top 10) -> Gộp RRF -> Rerank (Top 3)."""
    start_time = time.time()
    
    # 1. Truy xuất song song
    vec_chunks, vec_urls = get_vector_results(query, k=top_k_initial)
    bm25_chunks, bm25_urls = get_bm25_results(query, k=top_k_initial)
    
    # 2. Gộp kết quả loại bỏ trùng lặp (Tối đa hóa Recall)
    merged_chunks, merged_urls = rrf_merge(vec_chunks, vec_urls, bm25_chunks, bm25_urls)
    
    # 3. Lọc nhiễu bằng Re-ranker (Tối đa hóa Precision)
    final_chunks, final_urls = local_rerank(query, merged_chunks, merged_urls, top_n=top_k_final)
    
    latency = time.time() - start_time
    
    return {
        "method": "Parallel Hybrid (Vector+BM25) + Rerank",
        "retrieved_chunks": final_chunks,
        "retrieved_urls": final_urls,
        "latency": latency
    }

In [5]:
evaluation_logs = []

print("BẮT ĐẦU CHẠY KIỂM THỬ HỆ THỐNG MỚI...")
for index, row in gt_df.iterrows():
    query = row['Query']
    true_tier = row['Tier']
    expected_urls = row['Expected_Source_URLs']
    
    print(f"Đang xử lý [{index+1}/{len(gt_df)}] (Tier {true_tier}): {query[:50]}...")
    
    # Chạy chiến lược mới
    res = advanced_hybrid_rag(query)
    
    # Ghi log
    log_entry = {
        "query_id": index + 1,
        "query": query,
        "true_tier": true_tier,
        "expected_urls": expected_urls,
        "hybrid_pipeline": {
            "method": res['method'],
            "latency": res['latency'],
            "retrieved_urls": res['retrieved_urls'],
            "chunks": res['retrieved_chunks']
        }
    }
    evaluation_logs.append(log_entry)

# Lưu kết quả
with open('retrieval_experiment_logs_V2.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_logs, f, ensure_ascii=False, indent=4)

print("HOÀN TẤT! Đã lưu kết quả vào 'retrieval_experiment_logs_V2.json'.")

BẮT ĐẦU CHẠY KIỂM THỬ HỆ THỐNG MỚI...
Đang xử lý [1/28] (Tier 1): Trường Đại học Công nghiệp TP.HCM có những giá trị...
Đang xử lý [2/28] (Tier 1): Điểm TOEIC 4 kỹ năng tối thiểu là bao nhiêu để đạt...
Đang xử lý [3/28] (Tier 1): Tài khoản Microsoft của sinh viên được yêu cầu xác...
Đang xử lý [4/28] (Tier 1): Học kỳ đầu tiên tân sinh viên có phải tự đăng ký h...
Đang xử lý [5/28] (Tier 1): Các ngân hàng nào hỗ trợ nộp học phí qua quầy giao...
Đang xử lý [6/28] (Tier 1): Để nhận học bổng doanh nghiệp, sinh viên cần đáp ứ...
Đang xử lý [7/28] (Tier 1): Cơ sở chính của trường được đặt tại địa chỉ nào?...
Đang xử lý [8/28] (Tier 1): Phòng Công tác chính trị và Hỗ trợ sinh viên chịu ...
Đang xử lý [9/28] (Tier 2): Sinh viên sẽ bị xử lý như thế nào nếu nộp chứng ch...
Đang xử lý [10/28] (Tier 2): Để được xét tốt nghiệp, sinh viên cần tích lũy đủ ...
Đang xử lý [11/28] (Tier 2): Sự khác biệt giữa mức học bổng của sinh viên đạt l...
Đang xử lý [12/28] (Tier 2): Sinh viên bị cảnh báo kết quả h